# code-server — minimal notebook

Fetch, install, start. The scripts are pulled from the repo instead of being
embedded, so this notebook stays ~100 lines instead of 1,650.

**Runtime → Run all.** No credentials needed: with no ngrok token it falls back
to a Cloudflare quick tunnel.

> Colab's FAQ does not permit remote-desktop / remote-IDE use, and Colab warns
> when it detects one. Read this as a convenience for a host you control, not as
> something to run on Colab indefinitely.

## 1. Fetch the scripts

In [ ]:
import pathlib, stat, urllib.request

REF  = "main"                       # pin a commit SHA to bypass the raw CDN cache
REPO = "recursive-factory/code-server"
BASE = pathlib.Path("/content/cs")

for rel in ["lib/common.sh", "install.sh", "serve.sh", "supervise.sh", "stop.sh", "doctor.sh"]:
    url = f"https://raw.githubusercontent.com/{REPO}/{REF}/colab/{rel}"
    f = BASE / rel
    f.parent.mkdir(parents=True, exist_ok=True)
    f.write_text(urllib.request.urlopen(url, timeout=30).read().decode())
    if f.suffix == ".sh":
        f.chmod(f.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

print("fetched into", BASE, "-", REF)

## 2. Configure (optional)

Everything has a working default — edit only what you need.

| Variable | Default | Notes |
|---|---|---|
| `TUNNEL` | `auto` | ngrok when a token exists, else a Cloudflare quick tunnel (no account) |
| `CS_PORT` | auto | first free of `8080`/`8090`/`8443`/`9000`/`13337`. Pin one, e.g. `"9090"`. 8080 is not safe on Colab, where `datalab/web/app.js` sometimes owns it |
| `NGROK_DOMAIN` | — | reserved ngrok domain, so the URL survives restarts |
| `IDLE` | `1800` | shut down after this many seconds with no connections |
| `MAX_RUNTIME` | `39600` | hard stop at 11 h |

The resolved port is recorded in `$CS_STATE_DIR/port`, so `stop.sh`, the idle
watchdog and `doctor.sh` always operate on the port actually in use.


In [ ]:
import os

NGROK_TOKEN  = ""     # or a Colab Secret named NGROK_TOKEN
TUNNEL       = ""     # "" = auto | "ngrok" | "cloudflared" | "none"
NGROK_DOMAIN = ""     # reserved domain -> stable URL, e.g. "me.ngrok-free.app"
CS_PORT      = ""     # "" = auto (first free of 8080/8090/8443/9000/13337), or e.g. 9090
IDLE         = 1800   # stop after 30 min with no connections
MAX_RUNTIME  = 39600  # hard stop at 11 h

tok = NGROK_TOKEN.strip()
if not tok:
    try:
        from google.colab import userdata
        tok = (userdata.get("NGROK_TOKEN") or "").strip()
    except Exception:
        pass

os.environ.update(NGROK_TOKEN=tok, CS_TUNNEL=TUNNEL,
                  CS_IDLE_TIMEOUT=str(IDLE), CS_MAX_RUNTIME=str(MAX_RUNTIME),
                  CS_PORT=str(CS_PORT))
if NGROK_DOMAIN:
    os.environ["NGROK_DOMAIN"] = NGROK_DOMAIN

print("tunnel:", TUNNEL or "auto", "| port:", CS_PORT or "auto", "| ngrok token:", bool(tok))

## 3. Install (1–3 min on a fresh runtime)

In [ ]:
import os, subprocess
subprocess.run(["bash", str(BASE / "install.sh")], check=True,
               env=dict(os.environ, WITH_CLOUDFLARED="1"))

## 4. Start (returns immediately)

In [ ]:
import os, pathlib, subprocess

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
STATE.mkdir(parents=True, exist_ok=True)

# stop.sh kills any previous supervisor and reclaims the port, so this is re-runnable
subprocess.run(["bash", str(BASE / "stop.sh")], check=False,
               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
(STATE / "credentials.txt").unlink(missing_ok=True)

with open(STATE / "supervise.out", "w") as fh:
    proc = subprocess.Popen(["bash", str(BASE / "supervise.sh")], stdout=fh,
                            stderr=subprocess.STDOUT, start_new_session=True)
(STATE / "supervise.pid").write_text(str(proc.pid))
print("supervisor pid", proc.pid, "-> log:", STATE / "supervise.out")

## 5. Wait for the URL (live progress)

In [ ]:
import os, time, pathlib
from IPython.display import display, clear_output, HTML

STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
creds, log = STATE / "credentials.txt", STATE / "supervise.out"

for t in range(300):
    raw = creds.read_text().strip() if creds.exists() else ""
    info = dict(l.split("=", 1) for l in raw.splitlines() if "=" in l) if "url=" in raw else {}
    clear_output(wait=True)

    url = info.get("url")
    if url:
        pw = info.get("password")
        display(HTML(f'<p style="font-size:17px"><b>code-server</b> '
                     f'<a href="{url}" target="_blank">{url}</a></p>'
                     f'<p><b>password</b> <code>{pw}</code></p>'))
        print("tunnel:", info.get("tunnel"), "| state:", STATE)
        break

    tail = "\n".join(log.read_text().splitlines()[-8:]) if log.exists() else "(starting)"
    print(f"{t}s\n" + "-" * 56 + "\n" + tail)
    time.sleep(1)
else:
    print("no URL after 300s — see", log)

## 6. Stop

In [ ]:
import os, pathlib, subprocess
STATE = pathlib.Path(os.environ.get("CS_STATE_DIR", "/content/.code-server"))
pid = STATE / "supervise.pid"
if pid.exists():
    try:
        os.killpg(os.getpgid(int(pid.read_text().strip())), 15)
    except (ValueError, ProcessLookupError):
        pass
    pid.unlink(missing_ok=True)
subprocess.run(["bash", str(BASE / "stop.sh")], check=False)

Diagnostics: `!bash {BASE}/doctor.sh` · logs: `/content/.code-server/`

Prefer no network fetch? Use `code-server.ipynb`, which embeds the same scripts
(readable, ~1,650 lines) and works without a clone.